# 00 · Setup and test subgraph

Run this once per machine. It
1. installs the stack (Colab) and checks the GPU and DGL,
2. extracts the thesis graph from `ddi_graph_and_linegraph.zip` and reads **only the drug graph** (3,342 drugs), so it fits in standard Colab RAM,
3. builds a random-drug test subgraph for warm start / diagnostics / episodic, and a no-negatives version for cold start,
4. saves both under `WORK_ROOT/test_graphs/`.

**Colab:** pin the runtime first: *Runtime → Change runtime type → Runtime version 2026.07* (Python 3.12). DGL has no wheels for the default Python 3.13 runtime.
The test subgraph is only for checking that the code runs. Its numbers mean nothing.

In [7]:
ON_COLAB  = False
REPO_DIR  = "/home/ppcrniai/Documents/ddi-linegraph"
DATA_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_data"
WORK_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_runs"
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [8]:
import os, sys
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
        raise SystemExit(f"Repo not found at {REPO_DIR}. Upload or clone it there first.")
    !bash {REPO_DIR}/setup/colab_setup.sh
# the editable install is only picked up by a fresh interpreter; make it importable now
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [9]:
!python {REPO_DIR}/setup/verify_gpu.py

python: 3.11.14 | torch: 2.7.0+cu128
device: NVIDIA GeForce RTX 5090 Laptop GPU | capability: sm_120
GPU matmul OK, checksum: -98749.7
dgl: 2.1.0
/home/ppcrniai/Documents/ddi-linegraph/.venv/lib/python3.11/site-packages/dgl/dataloading/dataloader.py:1149: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(
ScatterSAGEConv vs dgl SAGEConv, max abs diff (CPU): 4.77e-07
ScatterSAGEConv on cuda: output (64, 8)

All checks passed.


In [10]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

  data_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_data
  work_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs
  raw_data_dir           /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data   (does not exist yet)
  drug_db                /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/Drug_META_DDIE.db   (does not exist yet)
  event_db               /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/event_META_DDIE_5foldLabel.db   (does not exist yet)
  graph_zip              /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip   (does not exist yet)
  graph_bin              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/ddi_graph_and_linegraph.bin   (does not exist yet)
  thesis_warm_ckpt_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_smart_fusion_whole_graph_checkpoints   (does not exist yet)
  thesis_hidden384_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_hidden384_experiment   (does not ex

/home/ppcrniai/Documents/ddi-linegraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test subgraph
Random drugs → every interaction (and sampled non-interacting pair) among them → directed line graph, same rule as the thesis (`backtracking=False`).

In [11]:
from ddi import graph as G
from ddi.data import save_graph_pair

G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
drug_graph = G.load_drug_graph(paths.graph_bin)
print(drug_graph)

SEED = 0
sub, lg = G.make_test_subgraph(drug_graph, n_drugs=N_TEST_DRUGS, seed=SEED)
save_graph_pair(os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin"), sub, lg)

# cold start uses the graph without the sampled non-interacting pairs
sub_cs, _ = G.make_test_subgraph(drug_graph, n_drugs=N_TEST_DRUGS, seed=SEED, drop_negatives=True)
dgl.save_graphs(os.path.join(paths.test_graph_dir, f"test_drug_graph_noneg_{N_TEST_DRUGS}.bin"), [sub_cs])

unzipping /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip -> /home/ppcrniai/Documents/ddi-linegraph/ddi_runs


FileNotFoundError: [Errno 2] No such file or directory: '/home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip'

In [ ]:
labels = sub.edata["label"]
counts = torch.bincount(labels)
present = (counts > 0).nonzero().flatten()
print(f"classes present: {len(present)} | with >=4 examples: {int((counts >= 4).sum())}")
print(f"label-0 pairs: {int(counts[0])} of {labels.numel()}")
print("META-DDIE groups present  common(1-175): %d  fewer(176-204): %d  rare(205-227): %d" % (
    int(((present >= 1) & (present <= 175)).sum()),
    int(((present >= 176) & (present <= 204)).sum()),
    int(((present >= 205) & (present <= 227)).sum())))
print(f"line graph: {lg.num_nodes():,} nodes, {lg.num_edges():,} edges "
      f"(~{lg.num_edges()*16/1e9:.2f} GB of edge indices)")